In [ ]:
import os
os.chdir(path_to_wd)

import random
import tempfile
import numpy as np  # type: ignore
import scanpy as sc  # type: ignore
import scvi  # type: ignore
import torch  # type: ignore
from rich import print  # type: ignore
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
import statsmodels.api as sm
import statsmodels.formula.api as smf
from tqdm.auto import tqdm
from sklearn.neighbors import KernelDensity
import scipy.stats as stats
from scipy.stats import linregress
from sklearn.neighbors import NearestNeighbors
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams['font.family']= 'Dejavu serif'
plt.rcParams['pdf.fonttype'] = 'truetype'
matplotlib.rcParams['figure.dpi'] = 100

sc.set_figure_params(figsize=(4, 4))

scvi.settings.seed = 0
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## scVIVA model

In [ ]:
adata_sp = sc.read("Reproducibility/Data/TREKKER/UC_TREKKER_RNA_ATAC_spatial.h5ad")

In [ ]:
samples = ['P02', 'P06']

for sample_id in samples:
    print(f"=================================================")
    print(f"Starting processing for specimen: {sample_id}")
    print(f"=================================================")

    puck = adata_sp[adata_sp.obs['specimen'] == sample_id].copy()
    puck.obs['lineage_w_clone'] = puck.obs['lineage_w_clone'].astype(str).replace(['Myeloid', 'TNK', 'B'], 'immune')
    puck.obs['lineage_w_clone'] = puck.obs['lineage_w_clone'].astype('category')

    puck_rna = puck[:, puck.var.feature_types == "Gene Expression"].copy() 
    puck_rna.layers['counts'] = puck_rna.X

    #########################################################
    # Train scVIVA
    #########################################################
    setup_kwargs = {
        "sample_key": "specimen",  # column in adata.obs that contains the individual slide ID
        "labels_key": "lineage_w_clone2",  # column in adata.obs that contains the cell type labels
        "cell_coordinates_key": 'X_spatial',  # spatial coordinates key in adata.obsm
        "expression_embedding_key": 'MultiVI_latent',  # expression embedding key in adata.obsm
    }

    print(f"Preprocessing {sample_id}...")
    scvi.external.SCVIVA.preprocessing_anndata(
        puck_rna,
        k_nn=20,  # number of nearest neighbors for spatial graph construction
        **setup_kwargs,
    )

    puck_rna.layers['counts'] = puck_rna.X

    scvi.external.SCVIVA.setup_anndata(
        puck_rna,
        layer="counts",  # adata layer that contains the raw counts
        batch_key="specimen",  # column in adata.obs that contains the batch covariate
        **setup_kwargs,
    )

    #=================================================
    print(f"Training model for {sample_id}...")
    nichevae = scvi.external.SCVIVA(puck_rna)

    nichevae.train(
        max_epochs=600,
        early_stopping=True,
        check_val_every_n_epoch=1,
        batch_size=512,
        plan_kwargs={
            "lr": 5e-4,
        },
    )

    save_path = f'Reproducibility/Results/TREKKER/scVIVA/{sample_id}/model'
    nichevae.save(save_path)
    print(f"Model for {sample_id} successfully saved to {save_path}\n")

## Variance decomposition

- P02

In [ ]:
puck = adata_sp[adata_sp.obs['specimen'] == 'P02'].copy()
puck.obs['lineage_w_clone'] = puck.obs['lineage_w_clone'].astype(str).replace(['Myeloid', 'TNK', 'B'], 'immune')
puck.obs['lineage_w_clone'] = puck.obs['lineage_w_clone'].astype('category')

puck_rna = puck[:, puck.var.feature_types == "Gene Expression"].copy() 
puck_rna.layers['counts'] = puck_rna.X

In [ ]:
setup_kwargs = {
        "sample_key": "specimen",  # column in adata.obs that contains the individual slide ID
        "labels_key": "lineage_w_clone2",  # column in adata.obs that contains the cell type labels
        "cell_coordinates_key": 'X_spatial',  # spatial coordinates key in adata.obsm
        "expression_embedding_key": 'MultiVI_latent',  # expression embedding key in adata.obsm
    }

scvi.external.SCVIVA.preprocessing_anndata(
    puck_rna,
    k_nn=20,  # number of nearest neighbors for spatial graph construction
    **setup_kwargs,
)

In [ ]:
dir_path = 'Reproducibility/Results/TREKKER/scVIVA/P02/model'
nichevae = scvi.external.SCVIVA.load(f"{dir_path}", puck_rna)

puck_rna.obsm["X_scVIVA_intrinsic"] = nichevae.get_latent_representation(puck_rna)
puck_rna.obsm["X_scVIVA_niche"] = nichevae.predict_niche_activation(puck_rna)

In [ ]:
# Extract epithelial cells
adata = puck_rna[puck_rna.obs["lineage"] == "LUM"].copy()
sig_df = pd.read_csv("Reproducibility/Results/VISIONR/UC_TREKKER_Malignant_signature_score_literature.txt", index_col=0, sep='\t')
adata.obs['Hypoxia'] = sig_df.loc[adata.obs_names, 'Hypoxia']

In [ ]:
#########################################################
# Niche_Gradient
#########################################################
X_niche_3d = adata.obsm['X_scVIVA_niche']
X_niche_mean = np.mean(X_niche_3d, axis=1)

hypoxia_dims = [15]
X_hypoxia = X_niche_mean[:, hypoxia_dims]

pca_hypoxia = PCA(n_components=1)
niche_gradient = pca_hypoxia.fit_transform(X_hypoxia).flatten()
VEGFA_exp = adata[:, 'VEGFA'].X.toarray().flatten() if hasattr(adata[:, 'VEGFA'].X, "toarray") else adata[:, 'VEGFA'].X.flatten()

if np.corrcoef(niche_gradient, VEGFA_exp)[0, 1] < 0:
    niche_gradient = -niche_gradient

col_name = f'niche_gradient_{hypoxia_dims[0]}'
adata.obs[col_name] = niche_gradient
    
df_plot = pd.DataFrame({
    'Niche_Gradient': niche_gradient,
    'VEGFA_Expression': VEGFA_exp,
    'Clone': adata.obs['lineage_w_clone'].values
})

top_clones = df_plot['Clone'].value_counts().nlargest(5).index
df_plot_sub = df_plot[df_plot['Clone'].isin(top_clones)]

target_clones = ['P02_clone_1', 'P02_clone_2']

for clone in target_clones:
    subset = df_plot_sub[df_plot_sub['Clone'] == clone]
    subset = subset.dropna(subset=['Niche_Gradient', 'VEGFA_Expression'])
    slope, intercept, r_value, p_value, std_err = linregress(subset['Niche_Gradient'], subset['VEGFA_Expression'])
    
    print(f"=== {clone} ===")
    print(f"Regression Coefficient (Slope): {slope:.6f}")
    print(f"Intercept:                      {intercept:.6f}")
    print(f"R-squared:                      {r_value**2:.6f}")
    print(f"P-value:                        {p_value:.4e}")
    print("-" * 20)

sns.set_theme(style="whitegrid")
g = sns.lmplot(
    data=df_plot_sub, 
    x='Niche_Gradient', 
    y='VEGFA_Expression', 
    hue='Clone',
    scatter_kws={'alpha': 0.1, 's': 5},
    line_kws={'linewidth': 3},          
    height=6, 
    aspect=1.5,
    palette='Spectral'
)

g.set_axis_labels("Hypoxia Niche Gradient", "VEGFA Expression")
g.fig.suptitle("Consistent VEGFA Upregulation Across Clones along Niche Gradient", y=1.02)
plt.show()

In [ ]:
#########################################################
# Endothelial density by Gaussian kernel
#########################################################
coords = puck_rna.obsm['X_spatial']
is_endo = (puck_rna.obs['lineage_w_clone'] == 'Endothelial').values
endo_coords = coords[is_endo]

kde = KernelDensity(kernel='gaussian', bandwidth=150).fit(endo_coords)
puck_rna.obs['vascular_potential'] = kde.score_samples(coords)

target_var = 'niche_gradient_15'
target_clone = 'P02_clone_1'

target_mask = puck_rna.obs['lineage'].isin(['LUM'])
df_base = puck_rna.obs[target_mask].copy()

for col in ['Hypoxia', 'niche_gradient_15']:
    if col in adata.obs.columns:
        df_base[col] = adata.obs[col].reindex(df_base.index).values

df_base = df_base[df_base['lineage_w_clone'] == target_clone].copy()

#########################################################
n_bins = 7

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(8, 6))
line_color = 'firebrick' if target_var == 'Hypoxia' else 'navy'

try:
    df_base['temp_bin'] = pd.qcut(
        df_base['vascular_potential'].rank(method='first'), 
        q=n_bins, 
        labels=False
    )
    sns.lineplot(
        data=df_base,
        x='temp_bin',
        y=target_var,
        marker='o',
        color=line_color,
        errorbar='se',
        ax=ax,
        linewidth=2.5,
        markersize=8
    )
    ax.set_title(f'{target_var} along Vascular Potential Gradient', fontsize=14, pad=15)
    ax.set_xlabel('Bin Index (Low → High Vascular)', fontsize=12)
    ax.set_ylabel(f'Mean {target_var}', fontsize=12)
    ax.set_xticks(range(n_bins))
    
except Exception as e:
    ax.set_title('Error in Binning', fontsize=14)
    ax.text(0.5, 0.5, f'Error details:\n{str(e)}', 
            ha='center', va='center', transform=ax.transAxes, color='red', fontsize=12)

plt.tight_layout()
plt.savefig("Reproducibility/Results/Plots/Slide-tags/FigureS5I_vascular_density.pdf")

In [ ]:
#########################################################
# Correlation between hypoxia and niche_gradient_15
#########################################################
k_neighbors = 30
coords = df_base[['x', 'y']].values

nn = NearestNeighbors(n_neighbors=k_neighbors).fit(coords)
_, indices = nn.kneighbors(coords)

df_base['Hypoxia_neigh_mean'] = df_base['Hypoxia'].values[indices].mean(axis=1)
df_base['niche_15_neigh_mean'] = df_base['niche_gradient_15'].values[indices].mean(axis=1)

# ==========================================
df_plot = df_base[['Hypoxia_neigh_mean', 'niche_15_neigh_mean']].dropna()
rho, p_val = spearmanr(df_plot['Hypoxia_neigh_mean'], df_plot['niche_15_neigh_mean'])

custom_reds_list = [
    "#ffffff",  # 0%: 
    "#fcbba1",  # 25%: 
    "#fb6a4a",  # 50%: 
    "#cb181d",  # 75%: 
    "#67000d"   # 100%: 
]
Custom_Reds_cmap = LinearSegmentedColormap.from_list('CustomRedsWhite', custom_reds_list)

# ==========================================
# 2D KDE Density Plot 
# ==========================================
plt.figure(figsize=(8, 7))

sns.kdeplot(
    data=df_plot,
    x='Hypoxia_neigh_mean',
    y='niche_15_neigh_mean',
    fill=True,
    thresh=0,
    levels=100,
    cmap=Custom_Reds_cmap,
    cbar=True,
    cbar_kws={'label': 'Density'}
)

plt.title(f'Neighborhood-Level Density (k={k_neighbors})\n'
          f'Spearman $\\rho$ = {rho:.3f} (p = {p_val:.2e})', fontsize=14)
plt.xlabel('Hypoxia Score (Neighborhood Mean)', fontsize=12)
plt.ylabel('Niche Gradient 15 (Neighborhood Mean)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.3, color='white')
plt.tight_layout()
plt.savefig("Reproducibility/Results/Plots/Slide-tags/FigureS5I_hypoxia_niche_corr.pdf")
plt.close()

Gene

In [ ]:
genes_to_test = adata.var_names
anova_results = []

for gene in tqdm(genes_to_test, desc="Processing Genes"):
    if hasattr(adata[:, gene].X, "toarray"):
        expr = adata[:, gene].X.toarray().flatten()
    else:
        expr = adata[:, gene].X.flatten()
        
    df_gene = pd.DataFrame({
        'Expression': expr,
        'Niche_Gradient': niche_gradient, 
        'Clone': adata.obs['lineage_w_clone'].values
    })
    
    df_gene['Niche_Bin'] = pd.cut(df_gene['Niche_Gradient'], bins=20, labels=False)
    
    df_pseudo = df_gene.groupby(['Clone', 'Niche_Bin']).agg(
        Mean_Expression=('Expression', 'mean'),
        Niche_Gradient_Mean=('Niche_Gradient', 'mean'),
        Cell_Count=('Expression', 'size')
    ).reset_index()
    
    df_pseudo = df_pseudo[df_pseudo['Cell_Count'] >= 5]
    if df_pseudo.empty: continue
        
    formula = "Mean_Expression ~ Niche_Gradient_Mean + C(Clone)"
    model = smf.ols(formula, data=df_pseudo).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    
    ss_total = anova_table['sum_sq'].sum()
    eta_sq_niche = anova_table.loc['Niche_Gradient_Mean', 'sum_sq'] / ss_total
    eta_sq_clone = anova_table.loc['C(Clone)', 'sum_sq'] / ss_total
    
    env_coef = model.params['Niche_Gradient_Mean']
    top_clone = df_pseudo.groupby('Clone')['Mean_Expression'].mean().idxmax()
    
    # 1. Signed Env_Effect
    signed_env_effect = eta_sq_niche if env_coef > 0 else -eta_sq_niche
    
    # 2. Signed Clone_Effect
    if top_clone == 'P02_clone_1':
        signed_clone_effect = eta_sq_clone
    elif top_clone == 'P02_clone_2':
        signed_clone_effect = -eta_sq_clone
    else:
        signed_clone_effect = 0
        
    anova_results.append({
        'Gene': gene,
        'Env_Effect(η2_Niche)': eta_sq_niche,
        'Clone_Effect(η2_Clone)': eta_sq_clone,
        'Signed_Env_Effect': signed_env_effect,
        'Signed_Clone_Effect': signed_clone_effect,
        'Top_Clone': top_clone,
        'Model_R2': model.rsquared
    })

df_anova = pd.DataFrame(anova_results)
print(df_anova)
df_anova.to_csv(f"{dir_path}/hypoxia_niche_anova_df_15.txt", sep='\t')


TF

In [ ]:
tmp_path = "Reproducibility/Results/LINGER/TREKKER/output/cell_population_TF_activity.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(adata.obs_names, TF_activity.columns)
adata_tmp = adata[cell_use,:].copy()
adata_TF = sc.AnnData(TF_activity.T.loc[adata_tmp.obs_names,:].copy(), obs=adata_tmp.obs)
adata_TF = adata_TF.copy()
sc.pp.scale(adata_TF, max_value=10)

genes_to_test = adata_TF.var_names
anova_results = []

for gene in tqdm(genes_to_test, desc="Processing Genes"):
    if hasattr(adata_TF[:, gene].X, "toarray"):
        expr = adata_TF[:, gene].X.toarray().flatten()
    else:
        expr = adata_TF[:, gene].X.flatten()
        
    df_gene = pd.DataFrame({
        'Expression': expr,
        'Niche_Gradient': niche_gradient, 
        'Clone': adata_TF.obs['lineage_w_clone'].values
    })
    
    df_gene['Niche_Bin'] = pd.cut(df_gene['Niche_Gradient'], bins=20, labels=False)
    
    df_pseudo = df_gene.groupby(['Clone', 'Niche_Bin']).agg(
        Mean_Expression=('Expression', 'mean'),
        Niche_Gradient_Mean=('Niche_Gradient', 'mean'),
        Cell_Count=('Expression', 'size')
    ).reset_index()
    
    df_pseudo = df_pseudo[df_pseudo['Cell_Count'] >= 5]
    if df_pseudo.empty: continue
        
    formula = "Mean_Expression ~ Niche_Gradient_Mean + C(Clone)"
    model = smf.ols(formula, data=df_pseudo).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    
    ss_total = anova_table['sum_sq'].sum()
    eta_sq_niche = anova_table.loc['Niche_Gradient_Mean', 'sum_sq'] / ss_total
    eta_sq_clone = anova_table.loc['C(Clone)', 'sum_sq'] / ss_total
    
    env_coef = model.params['Niche_Gradient_Mean']
    top_clone = df_pseudo.groupby('Clone')['Mean_Expression'].mean().idxmax()
    
    # 1. Signed Env_Effect
    signed_env_effect = eta_sq_niche if env_coef > 0 else -eta_sq_niche
    
    # 2. Signed Clone_Effect
    if top_clone == 'P02_clone_1':
        signed_clone_effect = eta_sq_clone
    elif top_clone == 'P02_clone_2':
        signed_clone_effect = -eta_sq_clone
    else:
        signed_clone_effect = 0
        
    anova_results.append({
        'Gene': gene,
        'Env_Effect(η2_Niche)': eta_sq_niche,
        'Clone_Effect(η2_Clone)': eta_sq_clone,
        'Signed_Env_Effect': signed_env_effect,
        'Signed_Clone_Effect': signed_clone_effect,
        'Top_Clone': top_clone,
        'Model_R2': model.rsquared
    })

df_anova = pd.DataFrame(anova_results)
print(df_anova)

df_anova.to_csv(f"{dir_path}/hypoxia_niche_anova_df_15_TF.txt", sep='\t')

- P06

In [ ]:
puck = adata_sp[adata_sp.obs['specimen'] == 'P06'].copy()
puck.obs['lineage_w_clone'] = puck.obs['lineage_w_clone'].astype(str).replace(['Myeloid', 'TNK', 'B'], 'immune')
puck.obs['lineage_w_clone'] = puck.obs['lineage_w_clone'].astype('category')

puck_rna = puck[:, puck.var.feature_types == "Gene Expression"].copy() 
puck_rna.layers['counts'] = puck_rna.X

In [ ]:
setup_kwargs = {
        "sample_key": "specimen",  # column in adata.obs that contains the individual slide ID
        "labels_key": "lineage_w_clone2",  # column in adata.obs that contains the cell type labels
        "cell_coordinates_key": 'X_spatial',  # spatial coordinates key in adata.obsm
        "expression_embedding_key": 'MultiVI_latent',  # expression embedding key in adata.obsm
    }

scvi.external.SCVIVA.preprocessing_anndata(
    puck_rna,
    k_nn=20,  # number of nearest neighbors for spatial graph construction
    **setup_kwargs,
)

In [ ]:
dir_path = 'Reproducibility/Results/TREKKER/scVIVA/P06/model'
nichevae = scvi.external.SCVIVA.load(f"{dir_path}", puck_rna)

puck_rna.obsm["X_scVIVA_intrinsic"] = nichevae.get_latent_representation(puck_rna)
puck_rna.obsm["X_scVIVA_niche"] = nichevae.predict_niche_activation(puck_rna)

In [ ]:
# Extract epithelial cells
adata = puck_rna[puck_rna.obs["lineage_w_clone"].isin(['P06_clone_1', 'P06_clone_2'])].copy()
sig_df = pd.read_csv("Reproducibility/Results/VISIONR/UC_TREKKER_Malignant_signature_score_literature.txt", index_col=0, sep='\t')
adata.obs['Stress'] = sig_df.loc[adata.obs_names, 'Stress_Barkley']

In [ ]:
niche_latent = adata.obsm['X_scVIVA_niche']
stress_scores = adata.obs['Stress'].values
n_cells, n_niches, n_dims = niche_latent.shape

results = []

for i in range(n_niches):
    for j in range(n_dims):
        vector = niche_latent[:, i, j]       
        corr, pval = spearmanr(stress_scores, vector)
        results.append({
            'niche_index': i,
            'latent_dim': j,
            'niche_dim_label': f"Niche{i}_Dim{j}",
            'correlation': corr,
            'p_value': pval
        })

df_corr = pd.DataFrame(results)
state_registry = nichevae.adata_manager.get_state_registry('labels')
cell_type_mapping = state_registry['categorical_mapping']

df_corr['cell_type'] = df_corr['niche_index'].map(lambda x: cell_type_mapping[x])

df_corr['abs_correlation'] = df_corr['correlation'].abs()
df_ranked = df_corr.sort_values('abs_correlation', ascending=False)

print("Top 10 Niche Dimensions correlated with Hypoxia:")
print(df_ranked.head(10))

corr_matrix = df_corr.pivot(index='cell_type', columns='latent_dim', values='correlation')

plt.figure(figsize=(12, 5))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, annot=False)
plt.title('Correlation between scVIVA Niche Dimensions and stress Score')
plt.xlabel('Latent Dimension')
plt.ylabel('Cell type')
plt.show()

In [ ]:
#########################################################
# Niche_Gradient
#########################################################
X_niche_3d = adata.obsm['X_scVIVA_niche']
X_niche_mean = np.mean(X_niche_3d, axis=1)

stress_dims = [19]
X_stress = X_niche_mean[:, stress_dims]

pca_stress = PCA(n_components=1)
niche_gradient = pca_stress.fit_transform(X_stress).flatten()
DUSP1_exp = adata[:, 'DUSP1'].X.toarray().flatten() if hasattr(adata[:, 'DUSP1'].X, "toarray") else adata[:, 'DUSP1'].X.flatten()

if np.corrcoef(niche_gradient, DUSP1_exp)[0, 1] < 0:
    niche_gradient = -niche_gradient

df_plot = pd.DataFrame({
    'Niche_Gradient': niche_gradient,
    'DUSP1_Expression': DUSP1_exp,
    'Clone': adata.obs['lineage_w_clone'].values
})

top_clones = df_plot['Clone'].value_counts().nlargest(5).index
df_plot_sub = df_plot[df_plot['Clone'].isin(top_clones)]

target_clones = ['P06_clone_1', 'P06_clone_2']

for clone in target_clones:
    subset = df_plot_sub[df_plot_sub['Clone'] == clone]
    subset = subset.dropna(subset=['Niche_Gradient', 'DUSP1_Expression'])
    slope, intercept, r_value, p_value, std_err = linregress(subset['Niche_Gradient'], subset['DUSP1_Expression'])
    
    print(f"=== {clone} ===")
    print(f"Regression Coefficient (Slope): {slope:.6f}")
    print(f"Intercept:                      {intercept:.6f}")
    print(f"R-squared:                      {r_value**2:.6f}")
    print(f"P-value:                        {p_value:.4e}")
    print("-" * 20)

sns.set_theme(style="whitegrid")
g = sns.lmplot(
    data=df_plot_sub, 
    x='Niche_Gradient', 
    y='DUSP1_Expression', 
    hue='Clone',
    scatter_kws={'alpha': 0.1, 's': 5},
    line_kws={'linewidth': 3},          
    height=6, 
    aspect=1.5,
    palette='Spectral'
)

g.set_axis_labels("Stress Niche Gradient", "DUSP1 Expression")
g.fig.suptitle("Consistent DUSP1 Upregulation Across Clones along Niche Gradient", y=1.02)
plt.show()

Gene

In [ ]:
genes_to_test = adata.var_names
anova_results = []

for gene in tqdm(genes_to_test, desc="Processing Genes"):
    if hasattr(adata[:, gene].X, "toarray"):
        expr = adata[:, gene].X.toarray().flatten()
    else:
        expr = adata[:, gene].X.flatten()
        
    df_gene = pd.DataFrame({
        'Expression': expr,
        'Niche_Gradient': niche_gradient, 
        'Clone': adata.obs['lineage_w_clone'].values
    })
    
    df_gene['Niche_Bin'] = pd.cut(df_gene['Niche_Gradient'], bins=20, labels=False)
    
    df_pseudo = df_gene.groupby(['Clone', 'Niche_Bin']).agg(
        Mean_Expression=('Expression', 'mean'),
        Niche_Gradient_Mean=('Niche_Gradient', 'mean'),
        Cell_Count=('Expression', 'size')
    ).reset_index()
    
    df_pseudo = df_pseudo[df_pseudo['Cell_Count'] >= 5]
    if df_pseudo.empty: continue
        
    formula = "Mean_Expression ~ Niche_Gradient_Mean + C(Clone)"
    model = smf.ols(formula, data=df_pseudo).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    
    ss_total = anova_table['sum_sq'].sum()
    eta_sq_niche = anova_table.loc['Niche_Gradient_Mean', 'sum_sq'] / ss_total
    eta_sq_clone = anova_table.loc['C(Clone)', 'sum_sq'] / ss_total
    
    env_coef = model.params['Niche_Gradient_Mean']
    top_clone = df_pseudo.groupby('Clone')['Mean_Expression'].mean().idxmax()
    
    # 1. Signed Env_Effect
    signed_env_effect = eta_sq_niche if env_coef > 0 else -eta_sq_niche
    
    # 2. Signed Clone_Effect
    if top_clone == 'P06_clone_1':
        signed_clone_effect = eta_sq_clone
    elif top_clone == 'P06_clone_2':
        signed_clone_effect = -eta_sq_clone
    else:
        signed_clone_effect = 0
        
    anova_results.append({
        'Gene': gene,
        'Env_Effect(η2_Niche)': eta_sq_niche,
        'Clone_Effect(η2_Clone)': eta_sq_clone,
        'Signed_Env_Effect': signed_env_effect,
        'Signed_Clone_Effect': signed_clone_effect,
        'Top_Clone': top_clone,
        'Model_R2': model.rsquared
    })

df_anova = pd.DataFrame(anova_results)
print(df_anova)

df_anova.to_csv(f"{dir_path}/stress_niche_anova_df_19.txt", sep='\t')

TF

In [ ]:
tmp_path = "Reproducibility/Results/LINGER/TREKKER/output/cell_population_TF_activity.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(adata.obs_names, TF_activity.columns)
adata_tmp = adata[cell_use,:].copy()
adata_TF = sc.AnnData(TF_activity.T.loc[adata_tmp.obs_names,:].copy(), obs=adata_tmp.obs)
adata_TF = adata_TF.copy()
sc.pp.scale(adata_TF, max_value=10)

genes_to_test = adata_TF.var_names
anova_results = []

for gene in tqdm(genes_to_test, desc="Processing Genes"):
    if hasattr(adata_TF[:, gene].X, "toarray"):
        expr = adata_TF[:, gene].X.toarray().flatten()
    else:
        expr = adata_TF[:, gene].X.flatten()
        
    df_gene = pd.DataFrame({
        'Expression': expr,
        'Niche_Gradient': niche_gradient, 
        'Clone': adata_TF.obs['lineage_w_clone'].values
    })
    
    df_gene['Niche_Bin'] = pd.cut(df_gene['Niche_Gradient'], bins=20, labels=False)
    
    df_pseudo = df_gene.groupby(['Clone', 'Niche_Bin']).agg(
        Mean_Expression=('Expression', 'mean'),
        Niche_Gradient_Mean=('Niche_Gradient', 'mean'),
        Cell_Count=('Expression', 'size')
    ).reset_index()
    
    df_pseudo = df_pseudo[df_pseudo['Cell_Count'] >= 5]
    if df_pseudo.empty: continue
        
    formula = "Mean_Expression ~ Niche_Gradient_Mean + C(Clone)"
    model = smf.ols(formula, data=df_pseudo).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    
    ss_total = anova_table['sum_sq'].sum()
    eta_sq_niche = anova_table.loc['Niche_Gradient_Mean', 'sum_sq'] / ss_total
    eta_sq_clone = anova_table.loc['C(Clone)', 'sum_sq'] / ss_total
    
    env_coef = model.params['Niche_Gradient_Mean']
    top_clone = df_pseudo.groupby('Clone')['Mean_Expression'].mean().idxmax()
    
    # 1. Signed Env_Effect
    signed_env_effect = eta_sq_niche if env_coef > 0 else -eta_sq_niche
    
    # 2. Signed Clone_Effect
    if top_clone == 'P06_clone_1':
        signed_clone_effect = eta_sq_clone
    elif top_clone == 'P06_clone_2':
        signed_clone_effect = -eta_sq_clone
    else:
        signed_clone_effect = 0 
        
    anova_results.append({
        'Gene': gene,
        'Env_Effect(η2_Niche)': eta_sq_niche,
        'Clone_Effect(η2_Clone)': eta_sq_clone,
        'Signed_Env_Effect': signed_env_effect,    
        'Signed_Clone_Effect': signed_clone_effect,  
        'Top_Clone': top_clone,
        'Model_R2': model.rsquared
    })

df_anova = pd.DataFrame(anova_results)
print(df_anova)

df_anova.to_csv(f"{dir_path}/stress_niche_anova_df_19_TF.txt", sep='\t')